In [2]:
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated, List
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langchain_community.tools import TavilySearchResults
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

memory = MemorySaver()

search_tool = TavilySearchResults(max_results=2)
tools = [search_tool]

llm = ChatGroq(model="openai/gpt-oss-20b")
llm_with_tools = llm.bind_tools(tools=tools)


class BasicState(TypedDict):
    messages: Annotated[List, add_messages]


def model(state: BasicState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def tools_router(state: BasicState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
        return "tools"
    else:
        return END


graph = StateGraph(BasicState)
graph.add_node(model, "model")
graph.add_node("tools", ToolNode(tools=tools))

graph.set_entry_point("model")
graph.add_conditional_edges("model", tools_router)

graph.add_edge("tools", "model")

app = graph.compile(checkpointer=memory, interrupt_before=["tools"])

/tmp/ipykernel_6084/2268304911.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import TavilySearchResults
/tmp/ipykernel_6084/2268304911.py:14: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [3]:
config = {"configurable": {
    "thread_id": 1
}}

events = app.stream({
    "messages": [HumanMessage(content="What is the current weather in Chennai?")]
}, config=config, stream_mode="values")

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current weather in Chennai?
================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (fc_4f53c07d-0198-49b5-ad14-335bab575a7e)
 Call ID: fc_4f53c07d-0198-49b5-ad14-335bab575a7e
  Args:
    query: current weather in Chennai


In [4]:
snapshot = app.get_state(config=config)
snapshot.next

('tools',)

In [5]:
events = app.stream(None, config, stream_mode="values")
for event in events:
    event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (fc_4f53c07d-0198-49b5-ad14-335bab575a7e)
 Call ID: fc_4f53c07d-0198-49b5-ad14-335bab575a7e
  Args:
    query: current weather in Chennai
================================= Tool Message =================================
Name: tavily_search_results_json

[{"title": "Weather in Chennai", "url": "https://www.weatherapi.com/", "content": "{\"location\": {\"name\": \"Chennai\", \"region\": \"Tamil Nadu\", \"country\": \"India\", \"localtime\": \"2026-11-25 17:54\"}, \"current\": {\"last_updated\": \"2026-11-25 17:45\", \"temp_c\": 27.0, \"temp_f\": 80.6, \"feelslike_c\": 29.8, \"feelslike_f\": 85.6, \"humidity\": 81, \"wind_kph\": 9.4, \"wind_mph\": 5.8, \"wind_dir\": \"ENE\", \"precip_mm\": 0.01, \"precip_in\": 0.0, \"chance_of_rain\": 24, \"chance_of_snow\": 0, \"uv\": 0.0, \"condition\": \"Patchy rain nearby\"}, \"forecast\": [{\"date\": \"2026-11-25\", \"maxtemp_c\": 